# 01. 원본 검사와 사과 단위 데이터 준비

원본 ZIP → 매니페스트 → 이미지 무결성 → group_no 분할 순서입니다. 같은 사과의 사진은 서로 다른 분할에 넣지 않습니다.

실행 결과는 저장하지 않았습니다. 아래 실행 스위치는 기본 `False`이며, 전체 실행만으로 학습·파일 생성·서버 검사가 시작되지 않습니다.

## 데이터 파트 노트북 순서

| 순서 | 노트북 | 작업 |
| --- | --- | --- |
| 01 | 데이터 준비 (이 노트북) | 매니페스트, 이미지 무결성, 사과 단위 분할 |
| 02 | `02_training_comparison.ipynb` | 모델·뷰 수 비교, 개선 실험, 결과 비교 |
| 03 | `03_virtual_brix_experiments.ipynb` | 당도 생성, 결합 실험, 상품등급 학습·CV·진단 |
| 04 | `04_model_verification.ipynb` | 평가 절차, 재사용 Test 진단, 패키지 검사 |
| 05 | `05_demo_bundles.ipynb` | 묶음 생성, 출처·SHA 검증, 당도 연결 |

- 프로젝트 Python 커널을 선택하고 위에서부터 읽는다. 의존성은 `requirements-inference.txt`, 그래프는 matplotlib, 커널은 ipykernel이 필요하다.
- 실행 셀의 `RUN_...`은 기본 `False`다. 필요한 단계만 켜고, 기존 증거를 덮지 않도록 새 출력 경로를 쓴다. GPU 학습은 원격 GPU 환경(`scripts/remote/`)에서 한다.
- 일회성 실험 함수는 03~05 노트북에 있다. 추론 서버·학습·테스트가 함께 쓰는 함수는 `src/data`, `src/models`, `src/training`, `data/sampling` 모듈에 둔다.

In [ ]:
from pathlib import Path
import os
import sys
import json

# 저장소 루트 또는 notebooks 폴더에서 실행할 수 있습니다.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'src/data').is_dir() and (p / 'configs').is_dir()), None)
if ROOT is None:
    raise RuntimeError('CQC 저장소 안에서 노트북을 열어 주세요.')
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print('작업 폴더:', ROOT)


## 매니페스트 생성

원본은 읽기만 하고 결과는 별도 폴더에 저장합니다. 기존 확정 데이터는 덮어쓰지 않습니다.

In [ ]:
from src.data.manifest import build_manifest, write_outputs

RAW_ROOT = Path('data/raw')
OUTPUT = Path('outputs/notebook-data-preparation')
RUN_MANIFEST = False
if RUN_MANIFEST:
    if OUTPUT.exists():
        raise FileExistsError(OUTPUT)
    rows, summary = build_manifest(RAW_ROOT)
    write_outputs(rows, summary, OUTPUT / 'manifest.csv', OUTPUT / 'manifest-summary.json')
    print(summary['dataset'])


## 이미지 무결성 검사

PNG/ZIP 오류와 중복을 확인합니다. 전체 원본을 읽으므로 시간이 걸립니다.

In [ ]:
from src.data.image_quality import scan_images

RUN_IMAGE_CHECK = False
if RUN_IMAGE_CHECK:
    image_report = scan_images(RAW_ROOT)
    print({key: image_report[key] for key in ('checked_images', 'corrupt_images', 'duplicate_sha256_groups')})


## 사과 단위 분할

Test를 학습·모델 선택에 사용하지 않습니다. 분할 결과는 확인 후에만 운영 데이터에 반영합니다.

In [ ]:
from src.data.split_groups import load_groups, assign_groups, build_summary, write_outputs as write_splits

RUN_SPLIT = False
if RUN_SPLIT:
    groups = load_groups(OUTPUT / 'manifest.csv')
    records = assign_groups(groups, seed=42, folds=5)
    split_summary = build_summary(records, seed=42, folds=5)
    if (OUTPUT / 'splits.csv').exists():
        raise FileExistsError(OUTPUT / 'splits.csv')
    write_splits(records, split_summary, OUTPUT / 'splits.csv', OUTPUT / 'split-summary.json')
    print(split_summary['split_groups'])
